# Whisper-Medium Classifier

Standalone classifier trained on **Whisper-medium encoder embeddings** (1024-dim mean-pool over 30s chunks), extracted by `encoder_comparison.ipynb` and cached to `{folder}_whisper_whole.csv`.

**Purpose:** produce a third base model (alongside `text_rf` and `wavlm_wp`) to wire into `fusion_text_wavlm.ipynb` later.

**Output granularity:** one proba per audio file. Candidate isolation in split; no cross-answer aggregation.

**Pipeline:**
1. Load cached `{folder}_whisper_whole.csv` (columns `filename, whisper_0 ... whisper_1023`).
2. Candidate-isolated train/test split (same convention as encoder_comparison / wavlm_comparison).
3. Train XGBoost + RandomForest heads.
4. Threshold sweeps (coarse 0.20-0.80 step 0.05, fine 0.30-0.70 step 0.01) + rec@P{60..90}.
5. Save per-file proba CSVs under `checkpoints_whisper/` for fusion reuse.

In [ ]:
# ============================================================
# CONFIG — edit this cell only
# ============================================================
from pathlib import Path

TRAIN_FOLDERS = ["audios2", "audios4"]
TEST_FOLDER   = "audios5"     # empty "" -> 20% candidate-grouped holdout

TEST_RATIO  = 0.20
RANDOM_SEED = 42

PREC_TARGETS = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

NB_DIR   = Path(".").resolve()
SAVE_DIR = NB_DIR / "checkpoints_whisper"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Train: {TRAIN_FOLDERS}  |  Test: {TEST_FOLDER or f'{int(TEST_RATIO*100)}% candidate holdout'}")
print(f"NB_DIR: {NB_DIR}")
print(f"Save:   {SAVE_DIR}")

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix, classification_report)

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)

LABEL_MAP = {
    "cheating":1,"read":1,"reading":1,"scripted":1,"yes":1,"1":1,1:1,
    "not cheating":0,"not_cheating":0,"spontaneous":0,"no":0,"0":0,0:0,"genuine":0,
}

In [ ]:
def candidate_id(filename):
    stem  = Path(filename).stem
    parts = stem.rsplit('_', 1)
    return parts[0] if len(parts) == 2 else stem

def load_gt(gt_path):
    gt = pd.read_csv(gt_path)
    fn_col  = next((c for c in gt.columns if c.lower() in ('filename','file','name')), gt.columns[0])
    lbl_col = next((c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth')), gt.columns[-1])
    gt = gt.rename(columns={fn_col: 'filename', lbl_col: 'label_raw'})
    gt['label_int'] = gt['label_raw'].map(
        lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    gt = gt[gt['label_int'].isin([0,1])][['filename','label_int']].copy()
    gt['label_int'] = gt['label_int'].astype(int)
    return gt

def threshold_sweep(proba, y):
    best_thr, best_f1 = 0.5, 0.0
    for thr in np.arange(0.20, 0.81, 0.02):
        f = f1_score(y, (proba >= thr).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_thr = f, thr
    return round(best_thr, 2), round(best_f1, 4)

def coarse_sweep(proba, y, lo=0.20, hi=0.80, step=0.05):
    rows = []
    for thr in np.arange(lo, hi + 1e-9, step):
        pred = (proba >= thr).astype(int)
        cm   = confusion_matrix(y, pred, labels=[0,1])
        rows.append(dict(thr=round(float(thr),2),
            prec=round(precision_score(y, pred, zero_division=0),4),
            rec =round(recall_score(y, pred, zero_division=0),4),
            f1  =round(f1_score(y, pred, zero_division=0),4),
            tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]), tn=int(cm[0,0])))
    return pd.DataFrame(rows)

def fine_sweep(proba, y, lo=0.30, hi=0.70, step=0.01):
    return coarse_sweep(proba, y, lo=lo, hi=hi, step=step)

def rec_at_prec_targets(proba, y, targets=PREC_TARGETS, min_tp=3):
    y = np.asarray(y)
    rows = []
    for t in np.arange(0.05, 0.991, 0.005):
        pred = (proba >= t).astype(int)
        tp   = int(((pred == 1) & (y == 1)).sum())
        if tp < min_tp: continue
        rows.append((float(t),
                     float(precision_score(y, pred, zero_division=0)),
                     float(recall_score(y, pred, zero_division=0))))
    out = {}
    for tp_target in targets:
        cands = [(thr, p, r) for thr, p, r in rows if p >= tp_target]
        if cands:
            best = max(cands, key=lambda x: x[2])
            out[f'rec@P{int(tp_target*100)}'] = round(best[2], 4)
            out[f'thr@P{int(tp_target*100)}'] = round(best[0], 3)
        else:
            out[f'rec@P{int(tp_target*100)}'] = 0.0
            out[f'thr@P{int(tp_target*100)}'] = None
    return out

print('Helpers ready.')

## 1. Scan folders + verify whisper caches exist

In [ ]:
all_names = list(dict.fromkeys(TRAIN_FOLDERS + ([TEST_FOLDER] if TEST_FOLDER else [])))

folders = []
for name in all_names:
    gt_path      = NB_DIR / f'{name}GT.csv'
    whisper_path = NB_DIR / f'{name}_whisper_whole.csv'
    missing = [p.name for p in (gt_path, whisper_path) if not p.exists()]
    if missing:
        print(f'  SKIP {name}: missing {missing}'); continue
    folders.append({'name': name, 'gt': gt_path, 'whisper': whisper_path})
    print(f"  OK   {name}: gt={gt_path.name}  whisper={whisper_path.name}")

assert folders, 'No usable folders found. Run encoder_comparison.ipynb Whisper extraction first.'

## 2. Build master index + candidate-isolated train/test split

In [ ]:
rows = []
for m in folders:
    gt = load_gt(m['gt'])
    gt['folder']       = m['name']
    gt['candidate_id'] = gt['filename'].map(candidate_id)
    rows.append(gt)
master = pd.concat(rows, ignore_index=True)
print(f'Master index: {len(master)} rows, {master["candidate_id"].nunique()} unique candidates')

if TEST_FOLDER:
    test_mask  = master['folder'] == TEST_FOLDER
    test_cands = set(master.loc[test_mask, 'candidate_id'])
    train_mask = (master['folder'].isin([n for n in TRAIN_FOLDERS if n != TEST_FOLDER])
                  & ~master['candidate_id'].isin(test_cands))
    dropped = ((master['folder'].isin([n for n in TRAIN_FOLDERS if n != TEST_FOLDER]))
               & master['candidate_id'].isin(test_cands)).sum()
    print(f'Explicit test folder: {TEST_FOLDER}  |  dropped {int(dropped)} leak rows from train')
else:
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=RANDOM_SEED)
    tr_idx, te_idx = next(gss.split(master, groups=master['candidate_id']))
    train_mask = np.zeros(len(master), dtype=bool); train_mask[tr_idx] = True
    test_mask  = np.zeros(len(master), dtype=bool); test_mask[te_idx]  = True
    print(f'GroupShuffleSplit on candidate_id (test_size={TEST_RATIO})')

train_files = set(master.loc[train_mask, 'filename'])
test_files  = set(master.loc[test_mask,  'filename'])
assert not (train_files & test_files), 'filename overlap'

train_cands = set(master.loc[master['filename'].isin(train_files), 'candidate_id'])
test_cands  = set(master.loc[master['filename'].isin(test_files),  'candidate_id'])
assert not (train_cands & test_cands), 'candidate overlap -- isolation broken'

y_tr_master = master[master['filename'].isin(train_files)][['filename','label_int']]
y_te_master = master[master['filename'].isin(test_files)][['filename','label_int']]
print(f'Train: {len(train_files)} files  ({int((y_tr_master.label_int==1).sum())} cheating / '
      f'{int((y_tr_master.label_int==0).sum())} honest)   {len(train_cands)} candidates')
print(f'Test:  {len(test_files)} files  ({int((y_te_master.label_int==1).sum())} cheating / '
      f'{int((y_te_master.label_int==0).sum())} honest)   {len(test_cands)} candidates')

## 3. Load Whisper embeddings + build X/y

In [ ]:
dfs = [pd.read_csv(m['whisper']) for m in folders]
feat_df   = pd.concat(dfs, ignore_index=True)
feat_cols = [c for c in feat_df.columns if c.startswith('whisper_')]
assert feat_cols, 'No whisper_* columns found in cache CSVs'
print(f'Whisper feature matrix: {feat_df.shape}   ({len(feat_cols)} dims)')

tr_df = feat_df.merge(y_tr_master, on='filename', how='inner')
te_df = feat_df.merge(y_te_master, on='filename', how='inner')
X_tr = tr_df[feat_cols].fillna(0).values
y_tr = tr_df['label_int'].values
X_te = te_df[feat_cols].fillna(0).values
y_te = te_df['label_int'].values
print(f'Shapes -> X_tr={X_tr.shape}  X_te={X_te.shape}')
print(f'Train cheating rate: {float((y_tr==1).mean()):.3f}  |  Test cheating rate: {float((y_te==1).mean()):.3f}')

## 4. Train XGBoost + RandomForest

In [ ]:
sc  = StandardScaler().fit(X_tr)
Xtr = sc.transform(X_tr)
Xte = sc.transform(X_te)
spw = float((y_tr==0).sum()) / max(float((y_tr==1).sum()), 1.0)
colsample = 0.3 if X_tr.shape[1] > 500 else 0.8
print(f'scale_pos_weight={spw:.3f}  colsample_bytree={colsample}')

results = {}

# ---- XGBoost ----
xgb_clf = xgb.XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.04,
    subsample=0.8, colsample_bytree=colsample, min_child_weight=3,
    scale_pos_weight=spw, eval_metric='logloss',
    early_stopping_rounds=30, random_state=RANDOM_SEED, device='cpu')
xgb_clf.fit(Xtr, y_tr, eval_set=[(Xte, y_te)], verbose=False)
proba_xgb = xgb_clf.predict_proba(Xte)[:, 1]

# ---- RandomForest ----
rf_clf = RandomForestClassifier(
    n_estimators=500, max_depth=None, min_samples_leaf=2,
    class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)
rf_clf.fit(Xtr, y_tr)
proba_rf = rf_clf.predict_proba(Xte)[:, 1]

results['xgb'] = dict(clf=xgb_clf, proba=proba_xgb)
results['rf']  = dict(clf=rf_clf,  proba=proba_rf)
print('Trained: xgb, rf')

## 5. Evaluate — best-F1 + sweeps + rec@P targets

In [ ]:
summary_rows = []
for name, r in results.items():
    proba   = r['proba']
    thr, f1 = threshold_sweep(proba, y_te)
    pred    = (proba >= thr).astype(int)
    cm      = confusion_matrix(y_te, pred, labels=[0,1])
    rec_p   = rec_at_prec_targets(proba, y_te)

    row = dict(
        tag=f'whisper_{name}', n_feat=X_tr.shape[1], thr=thr, f1=f1,
        precision=round(precision_score(y_te, pred, zero_division=0),4),
        recall=round(recall_score(y_te, pred, zero_division=0),4),
        accuracy=round(accuracy_score(y_te, pred),4),
        tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]), tn=int(cm[0,0]),
        **rec_p,
    )
    summary_rows.append(row)

    print('\n' + '='*70)
    print(f"  whisper_{name}   F1={f1:.4f}  P={row['precision']:.4f}  R={row['recall']:.4f}  thr={thr}")
    print('='*70)
    print(classification_report(y_te, pred, target_names=['honest','cheating'], digits=4, zero_division=0))
    print(f"  TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")
    print(f"  rec @P60={row['rec@P60']:.3f}  @P65={row['rec@P65']:.3f}  @P70={row['rec@P70']:.3f}  "
          f"@P75={row['rec@P75']:.3f}  @P80={row['rec@P80']:.3f}  @P85={row['rec@P85']:.3f}  @P90={row['rec@P90']:.3f}")

    # Sweeps
    coarse = coarse_sweep(proba, y_te)
    fine   = fine_sweep(proba,   y_te)
    coarse.to_csv(SAVE_DIR / f'sweep_coarse_whisper_{name}.csv', index=False)
    fine.to_csv(  SAVE_DIR / f'sweep_fine_whisper_{name}.csv',   index=False)
    print('\n-- FINE SWEEP (0.30 -> 0.70 step 0.01) --')
    print(fine.to_string(index=False))

## 6. Save per-file probas + models + summary (for fusion reuse)

In [ ]:
# Per-file proba CSVs — same schema as encoder_comparison pred files
for name, r in results.items():
    pd.DataFrame({
        'filename':  te_df['filename'].values,
        'label_int': y_te,
        'proba':     r['proba'],
    }).to_csv(SAVE_DIR / f'pred_whisper_{name}.csv', index=False)

# Models + scaler
joblib.dump(sc, SAVE_DIR / 'scaler_whisper.pkl')
results['xgb']['clf'].save_model(str(SAVE_DIR / 'model_whisper_xgb.json'))
joblib.dump(results['rf']['clf'], SAVE_DIR / 'model_whisper_rf.pkl')

# Summary
cmp_df = pd.DataFrame(summary_rows).sort_values('f1', ascending=False).reset_index(drop=True)
cmp_df.to_csv(SAVE_DIR / 'whisper_classifier_comparison.csv', index=False)

print('='*100)
print('  WHISPER CLASSIFIER COMPARISON  (sorted by F1)')
print('='*100)
base_cols = ['tag','n_feat','thr','f1','precision','recall','accuracy','tp','fp','fn','tn']
print(cmp_df[[c for c in base_cols if c in cmp_df.columns]].to_string(index=False))
rp_cols = [f'rec@P{int(p*100)}' for p in PREC_TARGETS if f'rec@P{int(p*100)}' in cmp_df.columns]
if rp_cols:
    print('\n' + '='*100)
    print('  RECALL @ PRECISION TARGETS')
    print('='*100)
    print(cmp_df[['tag','f1','precision','recall'] + rp_cols].to_string(index=False))
print('='*100)

summary = {
    'train_folders': TRAIN_FOLDERS,
    'test_folder':   TEST_FOLDER,
    'test_ratio':    TEST_RATIO if not TEST_FOLDER else None,
    'encoder':       'openai/whisper-medium',
    'input_dim':     int(X_tr.shape[1]),
    'n_train':       int(len(train_files)),
    'n_test':        int(len(test_files)),
    'prec_targets':  PREC_TARGETS,
    'results':       cmp_df.to_dict(orient='records'),
}
with open(SAVE_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\nSaved:")
print(f"  {SAVE_DIR/'pred_whisper_xgb.csv'}")
print(f"  {SAVE_DIR/'pred_whisper_rf.csv'}")
print(f"  {SAVE_DIR/'scaler_whisper.pkl'}")
print(f"  {SAVE_DIR/'model_whisper_xgb.json'}")
print(f"  {SAVE_DIR/'model_whisper_rf.pkl'}")
print(f"  {SAVE_DIR/'whisper_classifier_comparison.csv'}")
print(f"  {SAVE_DIR/'summary.json'}")